# E3 UNE-EN ISO 5801:2019 Fans Performance testing using standardized airways

## Introduction

The most important document in Europe, for the correct design and operation of a test bench of an axial fan is the **UNE-EN ISO 5801 Standard**, with the 2019 version as currently in effect. It is available at the UPC library in [this link](https://plataforma-aenormas-aenor-com.recursos.biblioteca.upc.edu/standard/UNE/N0061895).

In the [`E1`](E1_measurement_of_magnitudes.ipynb) notebook we learned the fundamental principles for measuring the main magnitudes in an axial fan: flow rate, pressure, temperature, torque, rotational speed.... In the [`E2`](./E2_uncertainty_management.ipynb) we used the `uncertainty` python package to extend the measurements to report the error. In the present notebook, we apply the guides provided by the Standard to specific test configurations, due to the presence of ducts, wall proximity, inlet/outlet disturbance, ...

We will process laboratory data from a standardized **Category A Inlet-Chamber Test Rig** and propagate measurement uncertainties to verify compliance with ISO 5801 Clause 17 quality criteria.

## Learning Objectives

- **Identify** the four standard **Fan Installation Categories** (A, B, C, D) and understand the definitions of **Fan Pressure ($p_f$)**, **Fan Dynamic Pressure ($p_{fd}$)**, and **Fan Static Pressure ($p_{fs}$)**.
- **Process** raw experimental data from an **ISO 5801 Type 2 / Category A Inlet Chamber** using a multi-nozzle flow meter.
- **Apply** the Python `uncertainties` package to calculate fan characteristic curves ($\Phi - \Psi$, $q_V - p_f$, $\eta_f$) complete with **error bars and 95% confidence bands**.
- **Evaluate** whether the experimental test results fulfill the **maximum allowable uncertainty limits** specified in ISO 5801 Table 13.

## Previous tasks

The ISO 5801 Standard is almost 150 pages long... You don't need to read it all, for the moment... Review to following clauses along with `E1` and `E2` notebooks

- [] Installation categories (Clauses 5 and 6)
- [] Fan pressures and power (Clauses 3 and 15)
- [] Flow rate measurement (Clause 12.5 and Appendix A)
- [] Uncertainty limits (Clause 17, tables 12 and 13) (already read in `E2 notebook`)
- [] Humid Air Properties (Clause 12.9.1, 12.9.2 and Annex H)


## Some simple questions

1. In the `E1` notebook flow rate was measured using an in-duct ISA 1932 nozzle. For the category A, an inlet chamber is preferred for testing low-pressure axial fan. What do you think is the reason for that? How does the large cross-sectional area of the chamber ($A_3 > 5 A_1$) simplify the measurement of inlet total pressure ($p_{sg1}$)?
    
    Answer:

    Minimizing permanent pressure loss: An in-duct ISA 1932 nozzle introduces a large unrecoverable pressure drop ($\approx 40\text{–}60\%$ of $\Delta p$) and requires long straight duct sections ($10D\text{–}20D$), which can exceed the static pressure capability of a low-pressure axial fan and prevent testing near free delivery ($p_f \approx 0$).

    By continuity ($v_3 = v_1 \frac{A_1}{A_3}$), when the chamber cross-sectional area $A_3$ is more than $5$ times the fan inlet area $A_1$, the air velocity inside the chamber is less than $20\%$ of the fan inlet velocity ($v_3 < 0.2 v_1$), making the dynamic pressure negligible ($p_{d3} = \frac{1}{2}\rho v_3^2 < 0.04\, p_{d1}$).

    ***

2. If a laboratory measurement yields an uncertainty in the fan performance of $u_\eta = 4\%$, what steps should be taken to reduce it?
   
   Answer:

   Since fan efficiency uncertainty combines the relative uncertainties of flow rate, fan pressure, and shaft power.

   * Identify the dominant contributor (sensitivity analysis): Evaluate individual uncertainty contributions to see whether $P_a$, $q_V$, or $p_f$ dominates.

   * Improve shaft power ($P_a$) measurement: Replace indirect motor calibration curves ($M(P_e)$) with a direct inline or reaction torque transducer and high-accuracy optical tachometer, or use a Class 0.2 precision power analyzer with calibrated motor efficiency data.

   * Use low-range, high-accuracy differential pressure transmitters (e.g., $\pm 0.1\%\text{ FS}$) matched to the actual pressure range rather than oversized gauges.

   * Increase sampling time & verify flow uniformity

   ***

3. In which cases should be the compressibility of air be considered for the fan performance measurement? 
   
   Answer:

   According to ISO 5801, air may be treated as incompressible ($\rho = \text{const.}$) only at low pressure ratios, whereas compressibility must be accounted for when, Pressure ratio threshold: The fan pressure ratio exceeds $\frac{p_{t2}}{p_{t1}} > 1.013$ (equivalently, total pressure rise $\Delta p_t > 1300\text{ Pa}$ at standard atmospheric pressure $101.3\text{ kPa}$), or Mach number $\text{Ma} > 0.1$.

   ***

4. Water vapor ($H_2O$, molar mass $\approx 18\text{ g/mol}$) is lighter than dry air ($N_2/O_2$ mixture, molar mass $\approx 29\text{ g/mol}$).
   1. At identical ambient temperature ($20^\circ\text{C}$) and barometric pressure ($101325\text{ Pa}$), is moist air at $70\%$ relative humidity **more dense** or **less dense** than completely dry air ($0\%$ humidity)?
   2. What systematic error (overestimation or underestimation) occurs in mass flow rate ($q_m$) and air power ($P_u$) if humidity is ignored and $R_{\text{dry}}$ is used instead of $R_{\text{wet}}$?
   
   Answer:

   1. Moist air at $70\%$ relative humidity is less dense than dry air.

   2. Mass flow rate ($q_m$): Since nozzle mass flow scales with $q_m = C_d A \varepsilon \sqrt{2 \rho \Delta p}$, overestimating $\rho$ causes an overestimation of mass flow rate $q_m$ (by $\approx \frac{1}{2}\frac{\Delta \rho}{\rho} \approx +0.25\%$).

   ***

## Tasks

We are now switching to the computation of the performance of an axial fan, similar to the one studied in the CFD notebooks. The main characteristics and ambient conditions are:

- Fan Diameter ($D$): $0.315\text{ m}$ ($A_2 = \pi D^2 / 4$).
- Multi-nozzle area (Figure A.2): $A_n = 0.0125\text{ m}^2$, discharge coefficient $\alpha = 0.985$ [27].
- Barometric Pressure: $p_a = 100500 \pm 100\text{ Pa}$.
- Ambient Temperature: $T_a = 20.0 \pm 0.2^\circ\text{C}$.
- Relative Humidity: $h_{\text{rel}} = 55.0 \pm 3.0\%$ ($0.55 \pm 0.03$).

Operating Points Data:
- `dp_nozzle` ($\Delta p$ across nozzles in Pa): $[450 \pm 0.5, 350 \pm 0.5, 250 \pm 0.5, 150 \pm 0.5, 60 \pm 0.5]$
- `p_chamber` ($p_{e3}$ gauge pressure in Pa, negative): $[-280 \pm 0.5, -220 \pm 0.5, -160 \pm 0.5, -100 \pm 0.5, -35 \pm 0.5]$
- `torque` ($T_r$ torque in N·m): $[0.42 \pm 0.005, 0.45 \pm 0.005, 0.46 \pm 0.005, 0.41 \pm 0.005, 0.35 \pm 0.005]$
- `RPM` (rotational speed in RPM): $[2276 \pm 2, 2274 \pm 2, 2275 \pm 2, 2275 \pm 2, 2276 \pm 2]$

We consider that there is no variation on temperature.

### Write a python script that makes the computation and plot the curves

1. First import the needed modules:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uncertainties.unumpy as unp
from uncertainties import ufloat
```
2. Define all the variables
3. Compute the saturation vapor pressure (Eq. (26)), the partial vapor pressure, the gas constant for wet air (Eq. (25)) and, finally, the density of humid air.
4. Compute the mass and volumetric flow rates
5. Compute fan static, dynamic and total pressure.
6. Compute power and efficiency (both static and total)
7. Check if the relative errors are compliant with ISO 5801 Standard limits (Table 13) 
8. Plot total pressure, power and efficiency against volumetric flow rate, with the error bars.
9. Compute and plot the head and flow coefficients (see [`E1`](../1_Design/D1_fundamentals_and_dimensional_analysis.ipynb) notebook), also with the error bars. 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uncertainties.unumpy as unp
from uncertainties import ufloat

# ==============================================================================
# 2. Define all variables (with uncertainties)
# ==============================================================================
D     = 0.315                         # Fan diameter [m]
A2    = np.pi * (D ** 2) / 4.0        # Fan outlet area [m^2]
An    = 0.0125                        # Multi-nozzle throat area [m^2]
alpha = 0.985                         # Nozzle discharge coefficient [-]

pa    = ufloat(100500.0, 100.0)       # Barometric pressure [Pa]
Ta_C  = ufloat(20.0, 0.2)             # Ambient temperature [°C]
Ta_K  = Ta_C + 273.15                 # Ambient temperature [K]
h_rel = ufloat(0.55, 0.03)            # Relative humidity [-]

# Operating points arrays with uncertainties
dp_nozzle = unp.uarray([450.0, 350.0, 250.0, 150.0, 60.0],   [0.5]*5)    # [Pa]
p_chamber = unp.uarray([-280.0, -220.0, -160.0, -100.0, -35.0], [0.5]*5)  # [Pa]
torque    = unp.uarray([0.42, 0.45, 0.46, 0.41, 0.35],        [0.005]*5)  # [N·m]
rpm       = unp.uarray([2276.0, 2274.0, 2275.0, 2275.0, 2276.0], [2.0]*5) # [rpm]

# ==============================================================================
# 3. Saturation vapor pressure, partial pressure, wet gas constant & density
# ==============================================================================
R_d = 287.05   # Specific gas constant for dry air [J/(kg·K)]
R_v = 461.50   # Specific gas constant for water vapor [J/(kg·K)]

# Magnus-Tetens / ISO 5801 Eq. (26) saturation vapor pressure [Pa]
p_sat = 610.78 * unp.exp((17.27 * Ta_C) / (Ta_C + 237.3))
p_v   = h_rel * p_sat  # Partial vapor pressure [Pa]

# ISO 5801 Eq. (25) gas constant for humid air [J/(kg·K)]
R_w = R_d / (1.0 - (p_v / pa) * (1.0 - R_d / R_v))

# Density of humid air [kg/m^3]
rho = pa / (R_w * Ta_K)

print(f"Saturation vapor pressure p_sat : {p_sat:.2f} Pa")
print(f"Partial vapor pressure p_v      : {p_v:.2f} Pa")
print(f"Wet air gas constant R_w        : {R_w:.3f} J/(kg·K)")
print(f"Humid air density rho           : {rho:.5f} kg/m^3\n")

# ==============================================================================
# 4. Mass and volumetric flow rates
# ==============================================================================
qm = alpha * An * unp.sqrt(2.0 * rho * dp_nozzle)  # Mass flow rate [kg/s]
qv = qm / rho                                      # Volumetric flow rate [m^3/s]

# ==============================================================================
# 5. Fan static, dynamic, and total pressure
# ==============================================================================
v2  = qv / A2                          # Mean fan outlet velocity [m/s]
pfs = -p_chamber                       # Fan static pressure (p_s2 - p_t1 = 0 - p_e3) [Pa]
pd2 = 0.5 * rho * (v2 ** 2)            # Fan outlet dynamic pressure [Pa]
pf  = pfs + pd2                        # Fan total pressure [Pa]

# ==============================================================================
# 6. Shaft power and efficiencies (static and total)
# ==============================================================================
omega    = rpm * (2.0 * np.pi / 60.0)  # Angular velocity [rad/s]
Pa_shaft = omega * torque              # Mechanical shaft power [W]
eta_s    = (qv * pfs) / Pa_shaft       # Fan static efficiency [-]
eta_t    = (qv * pf) / Pa_shaft        # Fan total efficiency [-]

# Store results in a DataFrame
df = pd.DataFrame({
    'dp_nozzle (Pa)': dp_nozzle,
    'qm (kg/s)':      qm,
    'qv (m^3/s)':     qv,
    'pfs (Pa)':       pfs,
    'pd2 (Pa)':       pd2,
    'pf (Pa)':        pf,
    'Pa (W)':         Pa_shaft,
    'eta_s (-)':      eta_s,
    'eta_t (-)':      eta_t
})

# ==============================================================================
# 7. Check relative errors against ISO 5801 Standard limits (Table 13, k=2 95%)
# ==============================================================================
def rel_u95(u_arr):
    return 2.0 * (unp.std_devs(u_arr) / np.abs(unp.nominal_values(u_arr))) * 100.0

df['U95_qv (%)']    = rel_u95(qv)      # ISO 5801 limit: <= 2.0%
df['U95_pf (%)']    = rel_u95(pf)      # ISO 5801 limit: <= 1.4%
df['U95_Pa (%)']    = rel_u95(Pa_shaft)# ISO 5801 limit: <= 2.5%
df['U95_eta_t (%)'] = rel_u95(eta_t)   # ISO 5801 limit: <= 3.2%

iso_check = pd.DataFrame({
    'qv (m^3/s)':         unp.nominal_values(qv),
    'U95_qv [<=2.0%]':    df['U95_qv (%)'],
    'U95_pf [<=1.4%]':    df['U95_pf (%)'],
    'U95_Pa [<=2.5%]':    df['U95_Pa (%)'],
    'U95_eta_t [<=3.2%]': df['U95_eta_t (%)'],
    'ISO 5801 Compliant': (
        (df['U95_qv (%)'] <= 2.0) &
        (df['U95_pf (%)'] <= 1.4) &
        (df['U95_Pa (%)'] <= 2.5) &
        (df['U95_eta_t (%)'] <= 3.2)
    )
})
print("=== ISO 5801 Clause 17 / Table 13 Relative Uncertainty Check (k=2, 95%) ===")
print(iso_check.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ==============================================================================
# 8. Plot Total Pressure, Shaft Power, and Efficiency vs Volumetric Flow Rate
# ==============================================================================
qv_nom, qv_err       = unp.nominal_values(qv),       unp.std_devs(qv)
pf_nom, pf_err       = unp.nominal_values(pf),       unp.std_devs(pf)
pfs_nom, pfs_err     = unp.nominal_values(pfs),      unp.std_devs(pfs)
Pa_nom, Pa_err       = unp.nominal_values(Pa_shaft), unp.std_devs(Pa_shaft)
etat_nom, etat_err   = unp.nominal_values(eta_t),    unp.std_devs(eta_t)
etas_nom, etas_err   = unp.nominal_values(eta_s),    unp.std_devs(eta_s)

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)

# Total & Static Pressure
axes[0].errorbar(qv_nom, pf_nom, xerr=qv_err, yerr=pf_err, fmt='o-', color='tab:blue',
                 capsize=4, label=r'Total Pressure $p_f \pm 1\sigma$')
axes[0].errorbar(qv_nom, pfs_nom, xerr=qv_err, yerr=pfs_err, fmt='s--', color='tab:cyan',
                 capsize=4, label=r'Static Pressure $p_{fs} \pm 1\sigma$')
axes[0].set_title(r'Category A Axial Fan Performance Curves ($D = 0.315\ \mathrm{m}$)')
axes[0].set_ylabel(r'Pressure $\mathrm{(Pa)}$')
axes[0].legend()

# Shaft Power
axes[1].errorbar(qv_nom, Pa_nom, xerr=qv_err, yerr=Pa_err, fmt='o-', color='tab:red',
                 capsize=4, label=r'Shaft Power $P_a \pm 1\sigma$')
axes[1].set_ylabel(r'Shaft Power $\mathrm{(W)}$')
axes[1].legend()

# Total & Static Efficiency
axes[2].errorbar(qv_nom, etat_nom * 100, xerr=qv_err, yerr=etat_err * 100, fmt='o-', color='tab:green',
                 capsize=4, label=r'Total Efficiency $\eta_t \pm 1\sigma$')
axes[2].errorbar(qv_nom, etas_nom * 100, xerr=qv_err, yerr=etas_err * 100, fmt='^--', color='tab:olive',
                 capsize=4, label=r'Static Efficiency $\eta_s \pm 1\sigma$')
axes[2].set_ylabel(r'Efficiency $\mathrm{(\%)}$')
axes[2].set_xlabel(r'Volumetric Flow Rate $q_V\ \mathrm{(m^3/s)}$')
axes[2].legend()

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# ==============================================================================
# 9. Compute and plot dimensionless Flow (phi) and Head (psi) coefficients
# ==============================================================================
phi = qv / (omega * (D ** 3))
psi = pf / (rho * (omega ** 2) * (D ** 2))

phi_nom, phi_err = unp.nominal_values(phi), unp.std_devs(phi)
psi_nom, psi_err = unp.nominal_values(psi), unp.std_devs(psi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(phi_nom, psi_nom, xerr=phi_err, yerr=psi_err, fmt='o-', color='tab:purple',
            linewidth=2, capsize=4, label=r'$\psi(\phi) \pm 1\sigma$')
ax.set_title(r'Dimensionless Head Coefficient $\psi$ vs. Flow Coefficient $\phi$')
ax.set_xlabel(r'Flow Coefficient $\phi = \frac{q_V}{\omega D^3}$')
ax.set_ylabel(r'Head Coefficient $\psi = \frac{p_f}{\rho \omega^2 D^2}$')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

### Alternative computation of density of humid air

The package [`pyfluids`](https://github.com/portyanikhin/pyfluids), that is a python wrapper of [`CoolProp`](https://coolprop.org/), can accurate and easily compute the density of [humid air](https://github.com/portyanikhin/pyfluids#methods-of-humidair-instances). Here follows an example. Compare your computation following IS0 5801 indications with the density computed with `pyfluids`. 

In [ ]:
from pyfluids import HumidAir, InputHumidAir

humid_air = HumidAir().with_state(
    InputHumidAir.temperature(35), # in Celsius degress 
    InputHumidAir.relative_humidity(70), # in % 
    InputHumidAir.pressure(101000) # absolute pressure in Pa
    ) 

print(f"Humid Air Density: {humid_air.density} kg/m³")

Humid Air Density: 1.1253592049303383 kg/m³


## Your project

With respect to your project:
- Specify which installation category (A, B, C or D) describes your project's fan mounting.
- Make a search in internet, articles, book, ... of similar test benches.
- Make an estimation of the dimensions of your test bench:
  - Length and diameter of conducts
  - Dimensions of flowmeter (or flowmeters if there are more than one)
  - Dimensions of chambers, if needed
  - Method of modification of flow rate (operating points)
- Draw a sketch. No need for now to be accurate. You can use [draw.oi](https://app.diagrams.net/) for instance